# Load IMDB data

In [1]:
import duckdb
import pandas as pd
import logging
import requests
import time

In [2]:
# Setup logging
logging.basicConfig(
    filename='pipeline.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Connect to DuckDB
try:
    con = duckdb.connect('data/imdb_movies.db')
    logging.info('Connected to DuckDB successfully')
except Exception as e:
    logging.error(f'Failed to connect to DuckDB: {e}')
    raise

# Load each IMDb file into pandas then into DuckDB
files = {
    'name_basics':      'data/name.basics.tsv.gz',
    'title_basics':     'data/title.basics.tsv.gz',
    'title_crew':       'data/title.crew.tsv.gz',
    'title_principals': 'data/title.principals.tsv.gz',
    'title_ratings':    'data/title.ratings.tsv.gz',
}

for table_name, filepath in files.items():
    try:
        logging.info(f'Loading {filepath}...')
        df = pd.read_csv(filepath, sep='\t', compression='gzip', low_memory=False, na_values='\\N')
        con.execute(f'DROP TABLE IF EXISTS {table_name}')
        con.execute(f'CREATE TABLE {table_name} AS SELECT * FROM df')
        row_count = con.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
        logging.info(f'Loaded {table_name} successfully — {row_count} rows')
        print(f'✓ {table_name}: {row_count:,} rows')
    except Exception as e:
        logging.error(f'Failed to load {table_name}: {e}')
        raise

# Filter title_basics to movies only
try:
    con.execute('''
        CREATE OR REPLACE TABLE movies AS
        SELECT *
        FROM title_basics
        WHERE titleType = 'movie'
    ''')
    movie_count = con.execute('SELECT COUNT(*) FROM movies').fetchone()[0]
    logging.info(f'Created movies table — {movie_count} rows')
    print(f'✓ movies (filtered): {movie_count:,} rows')
except Exception as e:
    logging.error(f'Failed to create movies table: {e}')
    raise

print('\nAll tables loaded successfully into imdb_movies.db')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ name_basics: 15,182,692 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ title_basics: 12,376,284 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ title_crew: 12,380,743 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ title_principals: 98,523,875 rows
✓ title_ratings: 1,650,900 rows
✓ movies (filtered): 741,038 rows

All tables loaded successfully into imdb_movies.db


In [6]:
# Your TMDB API key
API_KEY = '1867e8b00ca347a64e123634955aa0a4'

# Connect to existing DuckDB database
try:
    con = duckdb.connect('data/imdb_movies.db')
    logging.info('Connected to DuckDB for TMDB data load')
except Exception as e:
    logging.error(f'Failed to connect to DuckDB: {e}')
    raise

# Get only movies with at least 5000 votes
try:
    movie_ids = con.execute('''
        SELECT m.tconst 
        FROM movies m
        JOIN title_ratings r ON m.tconst = r.tconst
        WHERE CAST(r.numVotes AS INTEGER) >= 5000
    ''').df()['tconst'].tolist()
    logging.info(f'Fetching TMDB data for {len(movie_ids)} movies')
    print(f'Fetching TMDB data for {len(movie_ids):,} movies...')
except Exception as e:
    logging.error(f'Failed to fetch movie IDs: {e}')
    raise

# Fetch revenue and budget data from TMDB for each movie
results = []

for i, imdb_id in enumerate(movie_ids):
    try:
        # Step 1: Find TMDB ID using IMDb ID
        find_url = f'https://api.themoviedb.org/3/find/{imdb_id}'
        find_response = requests.get(find_url, params={
            'api_key': API_KEY,
            'external_source': 'imdb_id'
        })
        find_data = find_response.json()

        if not find_data.get('movie_results'):
            continue

        tmdb_id = find_data['movie_results'][0]['id']

        # Step 2: Get full movie details including revenue and budget
        details_url = f'https://api.themoviedb.org/3/movie/{tmdb_id}'
        details_response = requests.get(details_url, params={'api_key': API_KEY})
        details = details_response.json()

        results.append({
            'imdb_id':           imdb_id,
            'tmdb_id':           tmdb_id,
            'title':             details.get('title'),
            'revenue':           details.get('revenue'),
            'budget':            details.get('budget'),
            'popularity':        details.get('popularity'),
            'release_date':      details.get('release_date'),
            'original_language': details.get('original_language'),
        })

        # Log progress every 100 movies
        if (i + 1) % 100 == 0:
            logging.info(f'Fetched {i + 1} / {len(movie_ids)} movies')
            print(f'  Progress: {i + 1:,} / {len(movie_ids):,}')

        # Rate limit — TMDB allows 40 requests/second, sleep to be safe
        time.sleep(0.05)

    except Exception as e:
        logging.warning(f'Failed to fetch TMDB data for {imdb_id}: {e}')
        continue

# Convert results to DataFrame
tmdb_df = pd.DataFrame(results)

# Filter out rows where revenue is 0 or missing (TMDB returns 0 for unknown)
tmdb_df = tmdb_df[tmdb_df['revenue'] > 0]

logging.info(f'TMDB fetch complete — {len(tmdb_df)} movies with revenue data')
print(f'\n✓ TMDB fetch complete: {len(tmdb_df):,} movies with revenue data')

# Load into DuckDB
try:
    con.execute('DROP TABLE IF EXISTS tmdb_movies')
    con.execute('CREATE TABLE tmdb_movies AS SELECT * FROM tmdb_df')
    logging.info('tmdb_movies table created in DuckDB')
    print('✓ tmdb_movies loaded into DuckDB')
except Exception as e:
    logging.error(f'Failed to load tmdb_movies into DuckDB: {e}')
    raise

con.close()

Fetching TMDB data for 18,681 movies...
  Progress: 100 / 18,681
  Progress: 200 / 18,681
  Progress: 300 / 18,681
  Progress: 400 / 18,681
  Progress: 500 / 18,681
  Progress: 600 / 18,681
  Progress: 700 / 18,681
  Progress: 800 / 18,681
  Progress: 900 / 18,681
  Progress: 1,000 / 18,681
  Progress: 1,100 / 18,681
  Progress: 1,200 / 18,681
  Progress: 1,300 / 18,681
  Progress: 1,400 / 18,681
  Progress: 1,500 / 18,681
  Progress: 1,600 / 18,681
  Progress: 1,700 / 18,681
  Progress: 1,800 / 18,681
  Progress: 1,900 / 18,681
  Progress: 2,000 / 18,681
  Progress: 2,100 / 18,681
  Progress: 2,200 / 18,681
  Progress: 2,300 / 18,681
  Progress: 2,400 / 18,681
  Progress: 2,500 / 18,681
  Progress: 2,600 / 18,681
  Progress: 2,700 / 18,681
  Progress: 2,800 / 18,681
  Progress: 2,900 / 18,681
  Progress: 3,000 / 18,681
  Progress: 3,100 / 18,681
  Progress: 3,200 / 18,681
  Progress: 3,300 / 18,681
  Progress: 3,400 / 18,681
  Progress: 3,500 / 18,681
  Progress: 3,600 / 18,681
  Prog

In [9]:
# Create a base set of tconsts — only movies with 5000+ votes AND revenue data from TMDB
con.execute('''
    CREATE OR REPLACE TABLE final_movies AS
    SELECT m.*
    FROM movies m
    JOIN title_ratings r ON m.tconst = r.tconst
    JOIN tmdb_movies t ON m.tconst = t.imdb_id
    WHERE CAST(r.numVotes AS INTEGER) >= 5000
    AND t.revenue > 0
''')

# Filter all other tables to only those tconsts
con.execute('''
    CREATE OR REPLACE TABLE final_ratings AS
    SELECT r.* FROM title_ratings r
    WHERE r.tconst IN (SELECT tconst FROM final_movies)
''')

con.execute('''
    CREATE OR REPLACE TABLE final_crew AS
    SELECT c.* FROM title_crew c
    WHERE c.tconst IN (SELECT tconst FROM final_movies)
''')

con.execute('''
    CREATE OR REPLACE TABLE final_principals AS
    SELECT p.* FROM title_principals p
    WHERE p.tconst IN (SELECT tconst FROM final_movies)
''')

# Filter name_basics to only people in final_principals
con.execute('''
    CREATE OR REPLACE TABLE final_names AS
    SELECT n.* FROM name_basics n
    WHERE n.nconst IN (SELECT nconst FROM final_principals)
''')

# Check new row counts
tables = ['final_movies', 'final_ratings', 'final_crew', 'final_principals', 'final_names', 'tmdb_movies']
for table in tables:
    count = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table}: {count:,} rows')

final_movies: 11,552 rows
final_ratings: 11,552 rows
final_crew: 11,552 rows
final_principals: 247,905 rows
final_names: 88,457 rows
tmdb_movies: 11,552 rows


In [10]:
tables_to_export = ['final_movies', 'final_ratings', 'final_crew', 'final_principals', 'final_names', 'tmdb_movies']

for table in tables_to_export:
    con.execute(f"COPY {table} TO 'data/{table}.parquet' (FORMAT PARQUET)")
    print(f'✓ Exported {table}.parquet')

✓ Exported final_movies.parquet
✓ Exported final_ratings.parquet
✓ Exported final_crew.parquet
✓ Exported final_principals.parquet
✓ Exported final_names.parquet
✓ Exported tmdb_movies.parquet


In [11]:
tables_to_export = ['final_movies', 'final_ratings', 'final_crew', 'final_principals', 'final_names', 'tmdb_movies']

for table in tables_to_export:
    con.execute(f"COPY {table} TO 'data/{table}.csv' (FORMAT CSV)")
    print(f'✓ Exported {table}.csv')

✓ Exported final_movies.csv
✓ Exported final_ratings.csv
✓ Exported final_crew.csv
✓ Exported final_principals.csv
✓ Exported final_names.csv
✓ Exported tmdb_movies.csv


In [12]:
# title_ratings numerical features
print("=== title_ratings ===")
print(con.execute('''
    SELECT 
        ROUND(MIN(CAST(averageRating AS FLOAT)), 2) as min_rating,
        ROUND(MAX(CAST(averageRating AS FLOAT)), 2) as max_rating,
        ROUND(AVG(CAST(averageRating AS FLOAT)), 2) as mean_rating,
        ROUND(STDDEV(CAST(averageRating AS FLOAT)), 2) as std_rating,
        COUNT(*) - COUNT(averageRating) as missing_rating,
        MIN(CAST(numVotes AS INTEGER)) as min_votes,
        MAX(CAST(numVotes AS INTEGER)) as max_votes,
        ROUND(AVG(CAST(numVotes AS FLOAT)), 2) as mean_votes,
        ROUND(STDDEV(CAST(numVotes AS FLOAT)), 2) as std_votes,
        COUNT(*) - COUNT(numVotes) as missing_votes
    FROM final_ratings
''').df().to_string())

# final_movies numerical features
print("\n=== final_movies ===")
print(con.execute('''
    SELECT
        MIN(TRY_CAST(startYear AS INTEGER)) as min_year,
        MAX(TRY_CAST(startYear AS INTEGER)) as max_year,
        ROUND(AVG(TRY_CAST(startYear AS FLOAT)), 2) as mean_year,
        ROUND(STDDEV(TRY_CAST(startYear AS FLOAT)), 2) as std_year,
        COUNT(*) - COUNT(TRY_CAST(startYear AS INTEGER)) as missing_year,
        MIN(TRY_CAST(runtimeMinutes AS INTEGER)) as min_runtime,
        MAX(TRY_CAST(runtimeMinutes AS INTEGER)) as max_runtime,
        ROUND(AVG(TRY_CAST(runtimeMinutes AS FLOAT)), 2) as mean_runtime,
        ROUND(STDDEV(TRY_CAST(runtimeMinutes AS FLOAT)), 2) as std_runtime,
        COUNT(*) - COUNT(TRY_CAST(runtimeMinutes AS INTEGER)) as missing_runtime
    FROM final_movies
''').df().to_string())

# tmdb_movies numerical features
print("\n=== tmdb_movies ===")
print(con.execute('''
    SELECT
        MIN(revenue) as min_revenue,
        MAX(revenue) as max_revenue,
        ROUND(AVG(revenue), 2) as mean_revenue,
        ROUND(STDDEV(revenue), 2) as std_revenue,
        COUNT(*) - COUNT(revenue) as missing_revenue,
        MIN(budget) as min_budget,
        MAX(budget) as max_budget,
        ROUND(AVG(budget), 2) as mean_budget,
        ROUND(STDDEV(budget), 2) as std_budget,
        COUNT(*) - COUNT(budget) as missing_budget,
        ROUND(MIN(popularity), 2) as min_popularity,
        ROUND(MAX(popularity), 2) as max_popularity,
        ROUND(AVG(popularity), 2) as mean_popularity,
        ROUND(STDDEV(popularity), 2) as std_popularity,
        COUNT(*) - COUNT(popularity) as missing_popularity
    FROM tmdb_movies
''').df().to_string())

=== title_ratings ===
   min_rating  max_rating  mean_rating  std_rating  missing_rating  min_votes  max_votes  mean_votes  std_votes  missing_votes
0         1.3         9.3         6.51        0.99               0       5006    3169184    90657.65  183550.86              0

=== final_movies ===
   min_year  max_year  mean_year  std_year  missing_year  min_runtime  max_runtime  mean_runtime  std_runtime  missing_runtime
0      1915      2026    2002.78     17.65             0           43          566        109.62         22.0                0

=== tmdb_movies ===
   min_revenue  max_revenue  mean_revenue   std_revenue  missing_revenue  min_budget  max_budget  mean_budget   std_budget  missing_budget  min_popularity  max_popularity  mean_popularity  std_popularity  missing_popularity
0            1   2923706026   64527437.42  1.535492e+08                0           0   489900000  22398748.18  38003194.16               0             0.0          309.56             3.87            8.38